<a href="https://colab.research.google.com/github/UMA-GitHub-alt/mbed-os/blob/master/%E7%A0%94%E7%A9%B6%E5%AE%A4%E8%AA%B2%E9%A1%8C1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import transforms
from torch.optim.lr_scheduler import StepLR

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5), # 50%の確率で画像を左右反転
    transforms.RandomRotation(degrees=10),  # ±10度の範囲でランダムに回転
    transforms.ToTensor(),                  # 最後にTensor型に変換（必須）
])

In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=train_transform,
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [ ]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle = True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
import torch.nn as nn

class ConvNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # 1. 畳み込み層とプーリング層による特徴抽出器
        self.features = nn.Sequential(
            # 入力: (チャンネル数1, 縦28, 横28) -> モノクロ画像を想定
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 画像サイズが半分になる (32, 14, 14)

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # さらに半分になる (64, 7, 7)
        )

        # 2. 1次元に平坦化
        self.flatten = nn.Flatten()

        # 3. 全結合層による分類器
        # 64チャンネル * 縦7ピクセル * 横7ピクセル = 3136
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5), # 過学習を防ぐためにドロップアウトを追加（オプション）
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)     # 畳み込みで特徴を抽出
        x = self.flatten(x)      # 1次元のベクトルに変換
        logits = self.classifier(x) # 全結合層で10クラスに分類
        return logits

model = ConvNeuralNetwork().to(device)
print(model)

Using cuda device
ConvNeuralNetwork(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (classifier): Sequential(
    (0): Linear(in_features=3136, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 30
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
    scheduler.step()

    # 現在の学習率を表示（動作確認用）
    current_lr = scheduler.get_last_lr()[0]
    print(f"Current Learning Rate: {current_lr:.6f}\n")
print("Done!")

Epoch 1
-------------------------------
loss: 2.309291  [   64/60000]
loss: 0.766840  [ 6464/60000]
loss: 0.650281  [12864/60000]
loss: 0.785866  [19264/60000]
loss: 0.694806  [25664/60000]
loss: 0.604053  [32064/60000]
loss: 0.405668  [38464/60000]
loss: 0.499214  [44864/60000]
loss: 0.525769  [51264/60000]
loss: 0.507634  [57664/60000]
Test Error: 
 Accuracy: 85.3%, Avg loss: 0.424515 

Current Learning Rate: 0.001000

Epoch 2
-------------------------------
loss: 0.356894  [   64/60000]
loss: 0.514707  [ 6464/60000]
loss: 0.315174  [12864/60000]
loss: 0.507457  [19264/60000]
loss: 0.491557  [25664/60000]
loss: 0.368790  [32064/60000]
loss: 0.313917  [38464/60000]
loss: 0.540481  [44864/60000]
loss: 0.385577  [51264/60000]
loss: 0.239109  [57664/60000]
Test Error: 
 Accuracy: 86.5%, Avg loss: 0.362863 

Current Learning Rate: 0.001000

Epoch 3
-------------------------------
loss: 0.404938  [   64/60000]
loss: 0.350123  [ 6464/60000]
loss: 0.436756  [12864/60000]
loss: 0.397777  [192